# HydroPredict SHP preprocessing and missed-fault investigation

**Goal:** determine whether timestamps, channel ordering, scaling, or lost context
explain missed faults, then reproduce the recorded model experiments.

1. Upload this notebook to Colab. A **standard CPU runtime** is sufficient.
2. Run the cells in order and upload **hydropredict-investigation-2026-10-21.zip**
   when prompted. This is the new investigation package, not the older training ZIP.
3. Inspect the tables and plots, then run the final download cell to keep results.

The package contains a committed source snapshot, development-only telemetry,
saved baseline score evidence, and the frozen LSTM bundle. No laptop connection,
Google Drive authorisation, model training, or threshold tuning is required.

**Interpretation:** February-April 2019 has already been used for development.
There are nine recorded fault timestamps, six originally adequately observed faults,
and only three adequately scored faults in the common temporal comparison.
Overlapping fault windows are not independent failures. Normal references were
publisher selected. No operational accuracy or general false-alarm guarantee follows
from these results. Channel meanings, timezone evidence, and equipment IDs remain
unverified. No final-test observations are included.

## 1. Upload and verify the investigation package

Use only the trusted package generated from your repository. Hashes detect changed
files; they do not establish trust in an unknown package. A fresh temporary folder
is used for every execution. Rerun from this cell to start a new independent session.

In [ ]:
import hashlib
import io
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

try:
    from google.colab import files

    COLAB = True
except ImportError:
    COLAB = False

if COLAB:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly the investigation ZIP.")
    package_bytes = next(iter(uploaded.values()))
    del uploaded
else:
    # Optional local verification: set this environment variable to the package.
    package_bytes = Path(os.environ["HYDROPREDICT_INVESTIGATION_ZIP"]).read_bytes()

WORK = Path(tempfile.mkdtemp(prefix="hydropredict-investigation-"))


def extract_zip(payload, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for entry in archive.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
        archive.extractall(destination)


PACKAGE = WORK / "package"
extract_zip(package_bytes, PACKAGE)
manifest = json.loads((PACKAGE / "investigation_manifest.json").read_text())
assert manifest["test_data_included"] is False
for name, expected in manifest["files_sha256"].items():
    target = (PACKAGE / name).resolve()
    if not target.is_relative_to(PACKAGE.resolve()):
        raise ValueError("Unsafe manifest path")
    actual = hashlib.sha256(target.read_bytes()).hexdigest().upper()
    if actual != expected:
        raise ValueError(f"Changed package file: {name}")
PROJECT = WORK / "project"
extract_zip((PACKAGE / "project.zip").read_bytes(), PROJECT)
INPUT_ZIP = PACKAGE / "inputs/original-lstm-input.zip"
ORIGINAL = WORK / "original-inputs"
extract_zip(INPUT_ZIP.read_bytes(), ORIGINAL)
original_manifest = json.loads((ORIGINAL / "input_manifest.json").read_text())
assert original_manifest["test_data_included"] is False
for name, expected in original_manifest["files_sha256"].items():
    target = (ORIGINAL / name).resolve()
    assert target.is_relative_to(ORIGINAL.resolve())
    assert hashlib.sha256(target.read_bytes()).hexdigest().upper() == expected
EVIDENCE = PACKAGE / "evidence"
print("Package SHA256:", hashlib.sha256(package_bytes).hexdigest().upper())
print("Committed source:", manifest["source_commit"])
print("Model:", manifest["model_version"])
print("Workspace:", WORK)
del package_bytes

## 2. Install the snapshot and load development inputs

Colab normally has PyTorch installed. The LSTM extra installs it if necessary.
Inference explicitly uses CPU. These dependencies come from the normal package
registry; the source code comes from your verified committed snapshot.

In [ ]:
if COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT) + "[analysis,lstm]"],
        check=True,
    )
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
# Imports follow installation and source-path setup in this Colab cell.
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402

try:
    from IPython.display import Image, display
except ImportError:
    # Local non-Jupyter verification can still save all plots and tables.
    def display(value):
        print(value)

    def Image(filename):
        return filename


from hydropredict.analysis.readiness import fault_window_coverage  # noqa: E402
from hydropredict.data.preprocessing import StandardTelemetryScaler  # noqa: E402
from hydropredict.data.schema import MEASUREMENT_COLUMNS  # noqa: E402
from hydropredict.models.lstm import SHPLSTMAutoencoder  # noqa: E402

torch.set_num_threads(1)
CHANNELS = list(MEASUREMENT_COLUMNS)


def read_data(name):
    frame = pd.read_csv(ORIGINAL / "data" / (name + ".csv"))
    frame["t"] = pd.to_datetime(frame.t, utc=True, format="mixed")
    return frame


training = read_data("normal_training")
timeline = read_data("validation_context")
prefault = read_data("validation_prefault")
nominal = read_data("validation_nominal")
faults = read_data("evaluation_faults").t.tolist()
lower, upper = map(pd.Timestamp, original_manifest["validation_interval"])
assert training.t.max() < lower
assert timeline.t.between(lower, upper, inclusive="left").all()
assert all(lower <= fault < upper for fault in faults)
assert not timeline.t.duplicated().any() and timeline.t.is_monotonic_increasing
assert np.isfinite(timeline[CHANNELS].to_numpy()).all()
display(
    pd.DataFrame(
        [
            {
                "block": name,
                "rows": len(frame),
                "start_utc": frame.t.min(),
                "end_utc": frame.t.max(),
            }
            for name, frame in [
                ("normal_training", training),
                ("development_timeline", timeline),
                ("prefault_excerpts", prefault),
                ("nominal_reference", nominal),
            ]
        ]
    )
)
print("Fault timestamps:", len(faults), "| CPU PyTorch:", torch.__version__)

## 3. Select a fault and trace preprocessing

`FAULT_INDEX = 1` selects the missed February 3 event (zero-based). Change this
index and rerun the plotting cells to inspect another event. This does not change
the model, calibration, thresholds, or recorded experiment results.

The frozen LSTM consumes observed V1-V6 readings directly, with its stored scaler
and gap-safe sequences. It does not use an interpolated grid. The assertions below
test that particular path; passing them is not proof that every possible upstream
data issue or the source sensor semantics is correct.

In [ ]:
FAULT_INDEX = 1
RUN_NAME = "investigation-v1"
if not 0 <= FAULT_INDEX < len(faults):
    raise ValueError("Choose a fault index from 0 to " + str(len(faults) - 1))
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use a simple run name without slashes or dots.")
RESULTS = WORK / "results" / RUN_NAME
RESULTS.mkdir(parents=True, exist_ok=True)
fault = faults[FAULT_INDEX]
display(pd.DataFrame({"index": range(len(faults)), "fault_utc": faults}))

model = SHPLSTMAutoencoder.load(EVIDENCE / "frozen_bundle_copy")
fit_end = int(len(training) * 0.8)
stopping_start = int(fit_end * 0.9)
expected_scaler = StandardTelemetryScaler().fit(training.iloc[:stopping_start])
assert list(model.scaler.means) == CHANNELS
np.testing.assert_allclose(
    list(model.scaler.means.values()), list(expected_scaler.means.values()), rtol=1e-10, atol=1e-12
)
np.testing.assert_allclose(
    list(model.scaler.stds.values()), list(expected_scaler.stds.values()), rtol=1e-10, atol=1e-12
)
scaled = model.scaler.transform(timeline)
recovered = model.scaler.inverse_transform(scaled)
np.testing.assert_allclose(recovered[CHANNELS], timeline[CHANNELS], rtol=1e-12, atol=1e-10)
pd.testing.assert_series_equal(scaled.t, timeline.t)

# Prefault observations must survive unchanged in the reconstructed union.
joined = prefault.merge(
    timeline,
    on="t",
    how="left",
    suffixes=("_source", "_context"),
    validate="one_to_one",
    indicator=True,
)
assert joined["_merge"].eq("both").all()
np.testing.assert_allclose(
    joined[[c + "_source" for c in CHANNELS]].to_numpy(),
    joined[[c + "_context" for c in CHANNELS]].to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)

sequences = model.sequences(timeline)
positions = sequences.endpoints.input_end_position.to_numpy(dtype=int)
np.testing.assert_allclose(
    sequences.values[:, -1, :],
    scaled.iloc[positions][CHANNELS].to_numpy(dtype=np.float32),
    rtol=0,
    atol=0,
)
frozen_scores = model.score(timeline, device="cpu")
stored_scores = pd.read_csv(EVIDENCE / "lstm-scores.csv")
stored_scores["t"] = pd.to_datetime(stored_scores.t, utc=True, format="mixed")
assert frozen_scores.t.reset_index(drop=True).equals(stored_scores.t.reset_index(drop=True))
error_columns = [c + "_reconstruction_error" for c in CHANNELS]
np.testing.assert_allclose(
    frozen_scores[["anomaly_score", *error_columns]],
    stored_scores[["anomaly_score", *error_columns]],
    rtol=1e-4,
    atol=1e-6,
)
assert np.array_equal(frozen_scores.is_alert, stored_scores.is_alert)
prefix = model.score(timeline.iloc[:200], device="cpu")
np.testing.assert_allclose(
    prefix.anomaly_score,
    frozen_scores[frozen_scores.t.isin(prefix.t)].anomaly_score,
    rtol=1e-4,
    atol=1e-6,
)

audit = {
    "source_commit": manifest["source_commit"],
    "model_version": model.version,
    "scaler_matches_weight_fitting_only": True,
    "scaling_round_trip": True,
    "timestamps_unchanged": True,
    "prefault_measurements_preserved": True,
    "sequence_channel_order": True,
    "saved_score_replay": True,
    "alert_decisions_identical": True,
    "prefix_invariance": True,
    "sequence_accounting": sequences.accounting,
    "threshold": model.threshold,
    "test_evaluated": False,
}
(RESULTS / "preprocessing-audit.json").write_text(json.dumps(audit, indent=2))
display(
    pd.DataFrame(
        [{k: v for k, v in audit.items() if isinstance(v, bool) and k != "test_evaluated"}]
    ).T.rename(columns={0: "passed"})
)
display(pd.DataFrame([sequences.accounting]))

## 4. Inspect coverage before interpreting a miss

An event needs at least 80% of the expected five-minute slots in its preceding six
hours and a reading within ten minutes of the fault. Sequence warm-up can reduce
scored coverage. Missing scores are unavailable evidence, not normal predictions.

In [ ]:
coverage = pd.DataFrame(fault_window_coverage(prefault.t, faults)).rename(
    columns={"eligible": "original_eligible", "coverage_fraction": "original_coverage"}
)
scored_coverage = pd.DataFrame(fault_window_coverage(frozen_scores.t, faults))
coverage["lstm_scored_coverage"] = scored_coverage.coverage_fraction
coverage["lstm_scored_eligible"] = scored_coverage.eligible
display(
    coverage[
        [
            "fault_time",
            "original_coverage",
            "original_eligible",
            "lstm_scored_coverage",
            "lstm_scored_eligible",
        ]
    ]
)
coverage.to_csv(RESULTS / "coverage.csv", index=False)

## 5. Compare raw and scaled signals for the selected fault

Affine standardisation should preserve the shape of gradual changes. The bands
show the earlier normal calibration 0.1-99.9 percentiles, not validated physical
limits. Values inside a band can still change over time or violate relationships.

In [ ]:
mask = (timeline.t >= fault - pd.Timedelta(hours=6)) & (timeline.t < fault)
window = timeline.loc[mask]
window_scaled = scaled.loc[mask]
calibration = training.iloc[fit_end:]
cal_scaled = model.scaler.transform(calibration)
figure, axes = plt.subplots(6, 2, figsize=(14, 12), sharex=True)
for row, channel in enumerate(CHANNELS):
    for col, (values, reference, label) in enumerate(
        [(window, calibration, "Raw"), (window_scaled, cal_scaled, "Scaled")]
    ):
        axis = axes[row, col]
        y = values[channel].copy()
        y.loc[values.t.diff().dt.total_seconds().gt(450)] = np.nan
        axis.plot((values.t - fault).dt.total_seconds() / 3600, y, color="#237c70", lw=1.2)
        bounds = reference[channel].quantile([0.001, 0.999])
        axis.axhspan(bounds.iloc[0], bounds.iloc[1], color="#88b7a5", alpha=0.2)
        axis.set_ylabel(channel + " / " + label)
        axis.grid(alpha=0.15)
axes[-1, 0].set_xlabel("Hours before fault")
axes[-1, 1].set_xlabel("Hours before fault")
figure.suptitle(f"Raw to scaled trace: {fault.isoformat()} | physical units unverified")
figure.tight_layout(rect=(0, 0, 1, 0.97))
figure.savefig(RESULTS / f"raw-scaled-event-{FAULT_INDEX}.png", dpi=130)
plt.show()
trace = window.merge(window_scaled, on="t", suffixes=("_raw", "_scaled"))
trace.to_csv(RESULTS / f"raw-scaled-event-{FAULT_INDEX}.csv", index=False)

## 6. Inspect the frozen LSTM score and its channel errors

The threshold is fixed. Channel errors explain reconstruction mismatch, not the
cause of a fault. A single startup crossing is weaker evidence than a sustained
precursor, even if it meets the declared event rule.

In [ ]:
event_scores = frozen_scores[
    (frozen_scores.t >= fault - pd.Timedelta(hours=6)) & (frozen_scores.t < fault)
]
if event_scores.empty:
    print("No eligible LSTM sequences before this fault. Inspect coverage above.")
else:
    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot((event_scores.t - fault).dt.total_seconds() / 3600, event_scores.anomaly_score)
    axes[0].axhline(model.threshold, color="#b46936", ls="--", label="Frozen threshold")
    axes[0].set_xlabel("Hours before fault")
    axes[0].set_ylabel("Mean squared reconstruction error")
    axes[0].legend()
    peak = event_scores.loc[event_scores.anomaly_score.idxmax()]
    axes[1].bar(CHANNELS, [peak[c] for c in error_columns], color="#398e7e")
    axes[1].set_title("Channel errors at peak score")
    figure.tight_layout()
    figure.savefig(RESULTS / f"lstm-event-{FAULT_INDEX}.png", dpi=130)
    plt.show()
    print("Scored readings:", len(event_scores), "| Flagged:", int(event_scores.is_alert.sum()))
    print("Peak / threshold:", float(peak.anomaly_score / model.threshold))

## 7. Reproduce the fixed four-model investigation

This calls the repository's tested implementation rather than duplicating model
code in notebook cells. All new thresholds use earlier normal-only calibration.
The unchanged baselines are compared on identical available timestamps.

Run the four candidates once. Do not lower thresholds or search repeatedly until
the development results look favourable. A successful reproduction should find
that none of these four candidates improves covered fault detection. That negative
result is evidence, not a reason to hide a run. Rerunning this cell with an existing
output directory intentionally fails; choose a new RUN_NAME or start fresh.

In [ ]:
STUDY = RESULTS / "study"
environment = {**os.environ, "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
subprocess.run(
    [
        sys.executable,
        "scripts/investigate_signals.py",
        "--inputs",
        str(INPUT_ZIP),
        "--diagnostics",
        str(EVIDENCE),
        "--source-commit",
        manifest["source_commit"],
        "--output",
        str(STUDY),
    ],
    cwd=PROJECT,
    env=environment,
    check=True,
)
comparison = pd.read_csv(STUDY / "comparison.csv")
display(comparison)
display(Image(filename=str(STUDY / "event-score-comparison.png")))
print(
    "Nominal alert fractions are selected-reference measurements, "
    "not operational false-alarm rates."
)

## 8. Investigate score overlap and slow six-hour changes

This descriptive follow-up fits no new detector. It checks whether the missed-event
peaks overlap ordinary reference scores and compares six-hour slopes with selected
normal controls. Controls overlap in time: the reported tail fractions are **not
p-values, accuracy, or independent trials**. Inspect monthly burden to identify
instability across operating periods.

In [ ]:
OVERLAP = RESULTS / "overlap"
subprocess.run(
    [
        sys.executable,
        "scripts/diagnose_signal_overlap.py",
        "--study",
        str(STUDY),
        "--inputs",
        str(INPUT_ZIP),
        "--output",
        str(OVERLAP),
    ],
    cwd=PROJECT,
    env=environment,
    check=True,
)
trends = pd.read_csv(OVERLAP / "six-hour-trends.csv")
display(trends[pd.to_datetime(trends.fault_time, utc=True) == fault])
ranks = pd.read_csv(OVERLAP / "event-score-ranks.csv")
display(
    ranks[ranks.eligible].pivot(
        index="model", columns="fault_time", values="nominal_fraction_at_or_above_event_peak"
    )
)
monthly = pd.read_csv(STUDY / "monthly-nominal.csv")
display(monthly.pivot(index="model", columns="month", values="point_alert_fraction"))
print((OVERLAP / "notes.json").read_text())

## 9. Record your interpretation before proposing another experiment

- Did raw observations survive the timeline merge, scaling and sequence construction?
- Which missed events had sufficient context? Which lacked it?
- Does a signal change stay within ordinary numerical ranges?
- Are the changes unusual in normal control windows too?
- Did a method gain detections at a comparable alert burden and coverage?
- Are apparent detections isolated startup crossings or shared between overlapping faults?

The February 3 gradual rise is a reason to investigate multi-hour dynamics, **not**
proof of degradation. A new longer-horizon experiment needs a new written hypothesis,
normal-only calibration, explicit context-loss accounting, an unchanged comparison
denominator, and independent final evaluation before a general performance claim.
Do not reinterpret these exploratory results as production readiness.

## 10. Download the evidence

Download before the Colab runtime disconnects. The export contains audit results,
plots, CSVs, candidate research bundles, runtime versions, source identity and
checksums. It does not automatically upload anything to Google Drive. Model joblib
files should only be loaded from this trusted run. This notebook performs no
training or promotion of the deployed LSTM.

In [ ]:
import platform

import sklearn

export_metadata = {
    "source_commit": manifest["source_commit"],
    "run_name": RUN_NAME,
    "selected_fault_index": FAULT_INDEX,
    "model_version": model.version,
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "test_evaluated": False,
    "model_promoted": False,
}
(RESULTS / "runtime.json").write_text(json.dumps(export_metadata, indent=2))
(RESULTS / "input-manifest.json").write_text(json.dumps(manifest, indent=2))
hashes = {
    str(path.relative_to(RESULTS)): hashlib.sha256(path.read_bytes()).hexdigest().upper()
    for path in RESULTS.rglob("*")
    if path.is_file() and path.name != "export-checksums.json"
}
(RESULTS / "export-checksums.json").write_text(json.dumps(hashes, indent=2))
RESULT_ARCHIVE = Path(shutil.make_archive(str(WORK / (RUN_NAME + "-results")), "zip", RESULTS))
print("Export:", RESULT_ARCHIVE, "| bytes:", RESULT_ARCHIVE.stat().st_size)
if COLAB:
    files.download(str(RESULT_ARCHIVE))